# Land Use and Functional Layer Assignment

This notebook assigns spatial attributes to H3 hexagonal units using OpenStreetMap data extracted from a Geofabrik PBF file. For each hexagon in the Area of Interest (AOI), it computes:
- a **dominant land-use label** (residential, commercial, industrial, farmland, green, water, construction, other)
- **POI-based functional layer counts** (schools, hospitals, parks, transport hubs, commercial venues, etc.)
- a binary **highway presence indicator** (motorway / trunk)

Output is saved as a GeoPackage (`.gpkg`) for use in the case study notebooks.

**Inputs required:**
- A Geofabrik `.osm.pbf` extract for the target area
- A shapefile / GeoPackage defining the AOI boundary
- A GeoPackage of H3 hexagons at the desired resolution

See [Spatial Characterization of Urban Units](https://worldbank.github.io/eca-resilience/notebooks/Methodology_land_usage.html) for methodological details.

## 0. Environment Setup

Install required packages if not already present in the environment.

In [ ]:
%%capture
!pip install osmium
!pip install geopandas

In [ ]:
%%capture
!conda install -c conda-forge pyrosm -y

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import osmium
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely import wkb
from pyrosm import OSM

from land_use_utils import (
    assign_land_use,
    assign_poi_layers,
    assign_highway_layer,
    circle_gdf,
    download_and_prepare_landuse_polys,
    download_and_prepare_pois,
    download_and_prepare_buildings,
)

## 2. Configuration

Set paths for the target city. To run on a different AOI, update the variables below.

In [ ]:
city_name = "metro_manila"

# OSM data: PBF extract from Geofabrik (https://download.geofabrik.de)
path_PBF = "geofabrik_data/metro_manila.osm.pbf"

# AOI boundary shapefile (used to clip OSM data)
path_AOI_shape = "./shape_files/shape_metro_manila.gpkg"

# H3 hexagon tessellation for the AOI
path_AOI_h3 = "./spatial_tessellations/gdf_h3_res8_manila.gpkg"

# Output path for the enriched hexagon layer
path_output = f"./land_usage/land_usage_h3_res8_{city_name}.gpkg"

## 3. Load AOI and H3 Grid

In [ ]:
# Initialise the OSM reader on the PBF file
osm = OSM(path_PBF)

In [ ]:
# Load AOI boundary and apply a small buffer to avoid clipping edge hexagons
gdf_AOI = gpd.read_file(path_AOI_shape)
gdf_AOI = gdf_AOI.buffer(0.02)

# Load H3 hexagon tessellation
gdf_h3_AOI = gpd.read_file(path_AOI_h3)
native_crs = gdf_h3_AOI.crs

# Dissolve AOI to a single geometry for spatial filtering of OSM data
boundary_geom = gdf_AOI.union_all()

## 4. Data Extraction

Extract three OSM layers from the PBF file, clipped to the AOI boundary:
- **Polygons**: land-use, natural, and leisure areas
- **POIs**: points of interest (amenities, shops, transport, etc.)
- **Buildings**: typed building footprints (commercial, industrial, institutional)

In [ ]:
polys = download_and_prepare_landuse_polys(osm, boundary_geom)
polys["geometry"] = polys.geometry.buffer(0)   # fix potential invalid geometries

pois  = download_and_prepare_pois(osm, boundary_geom)

bldgs = download_and_prepare_buildings(osm, boundary_geom)
bldgs["geometry"] = bldgs.geometry.buffer(0)   # fix potential invalid geometries

In [ ]:
# Quick visual check: polygons (orange), buildings (black), POIs (blue dots)
ax = gpd.GeoSeries([boundary_geom]).plot(facecolor="lightgrey")
polys[polys["natural"] != "peninsula"].plot(ax=ax, color="orange", alpha=0.5)
bldgs.plot(ax=ax, color="black")
pois.plot(ax=ax, markersize=1)

## 5. Assign Land Use, POI Layers, and Highway Layer

Classification follows a three-step logic:
1. **Land use**: polygon overlay (primary) → building footprint fallback → "other"
2. **POI layers**: point-in-hex counts per functional category; parks via area overlap
3. **Highway**: binary flag for hexagons intersecting motorway or trunk roads

In [ ]:
# Step 1: land-use label (polygon-based + building fallback)
# MIN_SHARE: minimum polygon area share required to assign a label
# return_shares=True: attach per-class area share dicts for inspection
gdf_h3_LU = assign_land_use(
    gdf_h3_AOI, polys, bldgs,
    MIN_SHARE=0.001,
    MIN_SHARE_BLDG=0.0001,
    return_shares=True
)

# Step 2: POI-based functional layers
# MIN_SHARE_AREA: minimum fraction of hex area a park polygon must cover to be counted
gdf_h3_LU_POI = assign_poi_layers(gdf_h3_LU, polys, pois, MIN_SHARE_AREA=0.05)

# Step 3: highway presence flag (motorway and trunk only)
gdf_h3_LU_POI = assign_highway_layer(osm, gdf_h3_LU_POI, filters=["motorway", "trunk"])

# Drop intermediate columns used during classification
gdf_h3_LU_POI = gdf_h3_LU_POI.drop(
    ["lu_poly", "lu_bldg", "lu_poly_share", "lu_bldg_share", "index_right"],
    axis=1, errors="ignore"
)

## 6. Save Output

In [ ]:
gdf_h3_LU_POI.to_file(path_output)
print(f"Saved: {path_output}")
print(f"Hexagons: {len(gdf_h3_LU_POI):,}")
print("\nLand use distribution:")
print(gdf_h3_LU_POI["land_use"].value_counts())

## 7. Plot Land Use Map

In [ ]:
gdf_plot = gdf_h3_LU_POI.copy()

counts  = gdf_plot["land_use"].value_counts()
percent = (counts / len(gdf_plot) * 100).round(1)

landuse_colors = {
    "residential": "#F4A261",
    "commercial":  "#E76F51",
    "industrial":  "#6D597A",
    "construction": "darkgrey",
    "farmland":    "#90BE6D",
    "green":       "#2A9D8F",
    "water":       "#4A90E2",
    "other":       "#EEEEEE"
}

gdf_plot["color"] = gdf_plot["land_use"].map(landuse_colors).fillna("#EEEEEE")

fig, ax = plt.subplots(figsize=(8, 8))
gdf_plot.plot(color=gdf_plot["color"], linewidth=0, ax=ax)

legend_elements = [
    mpatches.Patch(
        facecolor=landuse_colors.get(cls, "#EEEEEE"),
        label=f"{cls} ({percent[cls]}%)"
    )
    for cls in counts.index
]

ax.legend(handles=legend_elements, loc="upper right", frameon=False, ncol=3)
ax.set_axis_off()
plt.tight_layout()
plt.show()